In [1]:
from pathlib import Path
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd

import xesmf as xe

import os
import sys

sys.path.append('/home/548/cd3022/repos/solar-nowcast/modules')
import sat_preprocess

import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe
from pyearthtools.data.time import Petdt
from pyearthtools.pipeline.operations.xarray.join import GeospatialTimeSeriesMerge
import site_archive_nci

In [2]:
start_date = datetime(2024, 1, 1)
end_date   = datetime(2024, 1, 5)

lat_min=-35
lat_max=-32
lon_min=148
lon_max=152

In [3]:
base_path = Path('/g/data/rv74/satellite-products/arc/der/himawari-ahi/solar/p1s/latest')

files = []

current = start_date
while current <= end_date:
    year  = current.year
    month = current.month
    day   = current.day

    file_path = base_path / f"{year}/{month:02d}/{day:02d}"
    
    if file_path.exists():  # important for missing days
        files.extend(file_path.rglob("*.nc"))
    
    current += timedelta(days=1)

In [4]:
def preprocess(ds):
    return ds.sel(
        latitude=slice(lat_min, lat_max),
        longitude=slice(lon_min, lon_max)
    )[['surface_global_irradiance', 'solar_elevation']]

ghi = xr.open_mfdataset(files, preprocess=preprocess)

In [5]:
rad_list = []

ch_list = [
    'B03',
    'B04',
    # 'B06',
    'B08',
    'B11',
    'B13',
    'B15'
]


for channel in ch_list:

    ds = sat_preprocess.read_himawari_channel(
        channel,
        start_date, end_date,
        lat_min, lat_max, lon_min, lon_max,
        coords=True
    )

    rad_list.append(ds)

/home/548/cd3022/repos/solar-nowcast/modules/sat_preprocess.py:116: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  return xr.open_mfdataset(files, preprocess=preprocess)
/home/548/cd3022/repos/solar-nowcast/modules/sat_preprocess.py:116: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  return xr.open_mfdataset(file

In [6]:
ref_ds = rad_list[-1]

interp_list = []

for ds in rad_list:
    if ds.sizes != ref_ds.sizes:
        ds = ds.interp(y=ref_ds.y, x=ref_ds.x)

    # Drop conflicting coords (except for reference)
    if ds is not ref_ds:
        ds = ds.drop_vars(["latitude", "longitude"], errors="ignore")

    interp_list.append(ds)

ds_rad = xr.merge(interp_list)

/jobfs/164618770.gadi-pbs/ipykernel_721138/2887343635.py:15: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_rad = xr.merge(interp_list)
/jobfs/164618770.gadi-pbs/ipykernel_721138/2887343635.py:15: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_rad = xr.merge(interp_list)


In [7]:
regridder = xe.Regridder(
    ds_rad,
    ghi,
    method="bilinear",
    reuse_weights=False
)

In [8]:
ds_rad_interp = regridder(ds_rad)

/opt/conda/envs/pet/lib/python3.11/site-packages/xarray/computation/apply_ufunc.py:450: PerformanceWarning: Regridding is increasing the number of chunks by a factor of 31.0, you might want to specify sizes in `output_chunks` in the regridder call. Default behaviour is to preserve the chunk sizes from the input (5, 193).
  result_vars[name] = func(*variable_args)
/opt/conda/envs/pet/lib/python3.11/site-packages/dask/array/routines.py:333: PerformanceWarning: Increasing number of chunks by factor of 35
  intermediate = blockwise(
/opt/conda/envs/pet/lib/python3.11/site-packages/xarray/computation/apply_ufunc.py:450: PerformanceWarning: Regridding is increasing the number of chunks by a factor of 16.0, you might want to specify sizes in `output_chunks` in the regridder call. Default behaviour is to preserve the chunk sizes from the input (10, 193).
  result_vars[name] = func(*variable_args)
/opt/conda/envs/pet/lib/python3.11/site-packages/dask/array/routines.py:333: PerformanceWarning: I

In [9]:
ghi_aligned, ds_rad_interp = xr.align(
    ghi,
    ds_rad_interp,
    join="inner"
)

ds = xr.merge([ghi_aligned, ds_rad_interp])

In [10]:
for n in range(1, 11):
    ds[f'surface_global_irradiance_t{n}'] = (
        ds['surface_global_irradiance'].shift(time=-n)
    )

In [12]:
df = ds.to_dataframe()
data = df.dropna()

In [13]:
data.shape

(8759353, 18)

In [14]:
data.to_parquet('/scratch/er8/cd3022/xgb_datasets/syd_radiances_2024-01-01.parquet')